# 08 - Introducción a JAX y comparativa con NumPy

En este notebook veremos:

- `jax.numpy` (**jnp**): arrays, operaciones básicas y compatibilidad con NumPy.
- **`jit`**: compilación just-in-time para acelerar funciones.
- **`grad`**: derivadas automáticas.
- **`vmap`**: vectorización automática.
- Comparativa simple de tiempos frente a NumPy (en CPU; si hay GPU, JAX puede acelerarse más).

> Nota: Si JAX no está instalado en tu entorno, puedes ejecutar solo las celdas de NumPy o instalar JAX en Colab.


## 1. Importaciones y verificación de JAX

In [1]:
try:
    import jax
    import jax.numpy as jnp
    print("JAX version:", jax.__version__)
    print("Backend:", jax.default_backend())
except Exception as e:
    print("JAX no disponible en este entorno:", e)

JAX version: 0.7.2
Backend: cpu


## 2. Arrays y operaciones básicas con `jnp`

In [2]:
try:
    x = jnp.array([1.0, 2.0, 3.0])
    y = jnp.array([0.5, 0.5, 0.5])
    print(x + y)
    print(jnp.sqrt(x))
except NameError:
    print("Salta esta celda si JAX no está disponible.")

[1.5 2.5 3.5]
[1.        1.4142135 1.7320508]


## 3. `jit`: compilación just-in-time

In [8]:
try:
    import time
    def f_np(a, b):
        return a * a + b

    import numpy as np
    a_np = np.random.rand(50_000_000)
    b_np = np.random.rand(50_000_000)

    # NumPy puro
    t0 = time.time()
    out_np = f_np(a_np, b_np)
    t1 = time.time()

    # JAX (primera vez compila, segunda es más rápida)
    import jax, jax.numpy as jnp
    @jax.jit
    def f_jax(a, b):
        return a * a + b

    a_j = jnp.array(a_np)
    b_j = jnp.array(b_np)
    t2 = time.time()
    out_j1 = f_jax(a_j, b_j).block_until_ready()
    t3 = time.time()
    out_j2 = f_jax(a_j, b_j).block_until_ready()
    t4 = time.time()

    print(f"NumPy: {t1 - t0:.4f}s")
    print(f"JAX (1ª con compilación): {t3 - t2:.4f}s")
    print(f"JAX (2ª ya compilado):    {t4 - t3:.4f}s")
    print("Igualdad aproximada:", np.allclose(np.array(out_j2), out_np))
except Exception as e:
    print("No se pudo ejecutar JAX jit:", e)

NumPy: 0.6318s
JAX (1ª con compilación): 0.3393s
JAX (2ª ya compilado):    0.0893s
Igualdad aproximada: True


## 4. `grad`: derivadas automáticas

In [9]:
try:
    import jax.numpy as jnp, jax
    def f(x):
        return jnp.sin(x) + x**2

    df = jax.grad(f)
    print("f'(2.0) ≈", df(2.0))
except Exception as e:
    print("No se pudo usar grad:", e)

f'(2.0) ≈ 3.5838532


## 5. `vmap`: vectorización de funciones

In [10]:
try:
    import jax.numpy as jnp, jax
    def escala(x, s):
        return s * x

    escala_v = jax.vmap(escala, in_axes=(0, None))
    xs = jnp.arange(5.0)
    print(escala_v(xs, 2.0))  # escala cada elemento por 2
except Exception as e:
    print("No se pudo usar vmap:", e)

[0. 2. 4. 6. 8.]


## 📝 Ejercicios

1) Implementa `f(x) = x^3 + 2x` y calcula `f'(3)` con `jax.grad`.  
2) Usa `jax.jit` para acelerar una función que calcule `a*b + a**2` y compara tiempos (1ª vs 2ª ejecución).  
3) Con `vmap`, aplica una función que calcule `x^2 + 1` a un vector de 10 elementos.
